# Mixed classical-quantum diagrams in PyZX

PyZX has two related ways to describe classical information. A circuit measurement and conditional gate use a symbolic Boolean phase. A ground marker is a separate graph-level connection for a discarded output. This notebook shows both without treating them as interchangeable.

In [ ]:
import pyzx as zx
from pyzx.symbolic import Poly
from pyzx.utils import VertexType

## Measurement and feedforward

The first qubit is measured into `c[0]`. An `X` correction on the second qubit is applied only when that bit is 1. PyZX represents the outcome and the conditional rotation with the same Boolean variable.

In [ ]:
qasm = """OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[1];
h q[0];
measure q[0] -> c[0];
if (c==1) x q[1];
"""
circuit = zx.Circuit.from_qasm(qasm)
graph = circuit.to_graph()
assert graph.var_registry.types["c[0]"] is True
print(circuit.gates)
zx.draw(graph)

`substitute_variables` returns a new graph by default. The conditional `X` spider has phase 0 when the measured bit is 0 and phase 1 (that is, an `X` gate) when it is 1.

In [ ]:
conditional = [
    v for v in graph.vertices()
    if graph.qubit(v) == 1 and graph.type(v) == VertexType.X
    and isinstance(graph.phase(v), Poly)
]
assert len(conditional) == 1
for outcome in (0, 1):
    branch = graph.substitute_variables({"c[0]": outcome})
    assert branch.phase(conditional[0]) == outcome
    print(f"c[0]={outcome}: correction phase = {branch.phase(conditional[0])} pi")
assert isinstance(graph.phase(conditional[0]), Poly)

## Ground markers on graphs

A ground marker denotes a connection to a discard/trace generator. We can add `DiscardBit` after feedforward to discard the classical wire. This places a ground marker in the graph, alongside the symbolic measurement variable.

In [ ]:
from pyzx.circuit.gates import DiscardBit

discarded_circuit = circuit.copy()
discarded_circuit.gates.append(DiscardBit(0))
discarded_graph = discarded_circuit.to_graph()
assert discarded_graph.is_hybrid()
assert len(discarded_graph.grounds()) == 1
assert discarded_graph.var_registry.types["c[0]"] is True
print("Grounded vertices:", discarded_graph.grounds())
zx.draw(discarded_graph)

When building a graph directly, a ground marker can instead be added to an existing ZX spider with `set_ground`, or at creation with `add_vertex(..., ground=True)`. It does not itself create a named Boolean result.

In [ ]:
hybrid = zx.Graph()
input_vertex = hybrid.add_vertex(VertexType.BOUNDARY, 0, 0)
first_spider = hybrid.add_vertex(VertexType.Z, 0, 1)
grounded_spider = hybrid.add_vertex(VertexType.Z, 0, 2, ground=True)
output_vertex = hybrid.add_vertex(VertexType.BOUNDARY, 0, 3)
for left, right in ((input_vertex, first_spider), (first_spider, grounded_spider), (grounded_spider, output_vertex)):
    hybrid.add_edge((left, right))
hybrid.set_inputs((input_vertex,))
hybrid.set_outputs((output_vertex,))
assert hybrid.is_ground(grounded_spider)
assert hybrid.grounds() == {grounded_spider}
assert hybrid.is_hybrid()
zx.draw(hybrid)

Spider fusion preserves the ground connection. The rewrite changes vertex identifiers, so inspect `grounds()` again after simplifying.

In [ ]:
simplified = hybrid.copy()
assert zx.spider_simp(simplified)
assert len(simplified.grounds()) == 1
assert simplified.is_hybrid()
print("Grounded vertices after fusion:", simplified.grounds())
zx.draw(simplified)

Ground-marked graphs are not currently supported by `tensorfy` or `extract_circuit`. Use symbolic outcomes for the supported circuit-to-graph measurement and feedforward workflow; use ground markers when working directly with ZX-ground diagrams. See the [representation guide](../doc/representations.rst) and [Borgna et al.](https://arxiv.org/abs/2109.06071) for background.